# Step 6：Skew 主导组合策略、对冲与 PnL 归因

本 Notebook 由 `code 8.12/skew_strategy.py` 合并而来。默认读取本项目 `00_config.ipynb` 的 Quadratic、Derivative Skew、Derivative Curvature 配置。

建议按 `00 → 01 → 02 → 03 → 04 → 05 → 06` 顺序运行；也可以在上游 CSV 已生成后单独运行本 Notebook。


## 当前模块参数值

参数默认值统一在 `00_config.ipynb` 设置；本 cell 只打印当前内核中的实际值。


In [ ]:
_module_parameter_names = ["SKEW_STRATEGY_OUTPUT_PATH", "VOL_MODEL", "MODEL_SKEW_TAG", "STRATEGY_MAX_OPTIONS", "STRATEGY_TARGET_SKEW_ABS", "STRATEGY_MONTHLY_SKEW_DIRECTION", "STRATEGY_HEDGE_THETA", "STRATEGY_POSITION_BOUND", "STRATEGY_MAX_DAILY_TRADE_PER_LEG", "STRATEGY_MIN_OPTION_VOLUME", "STRATEGY_USE_PREMIUM_MARGIN_FILTER", "STRATEGY_MIN_PREMIUM_MARGIN_RATIO", "STRATEGY_MAX_ABS_LOG_MONEYNESS", "STRATEGY_MAX_MODEL_IV", "STRATEGY_MIN_CANDIDATE_OPTIONS", "STRATEGY_MAX_CANDIDATE_EXPIRIES", "STRATEGY_MIN_DTE_DAYS", "STRATEGY_RIDGE", "STRATEGY_GROSS_POSITION_PENALTY", "STRATEGY_INTEGER_OPTION_POSITIONS", "STRATEGY_INTEGER_FUTURES_POSITIONS", "STRATEGY_OPTION_MULTIPLIER", "STRATEGY_OPTION_FEE_PER_CONTRACT", "STRATEGY_FUTURES_MULTIPLIER", "STRATEGY_FUTURES_FEE_RATE", "STRATEGY_FUTURES_MARGIN_RATE", "STRATEGY_OPTION_MARGIN_RATE", "STRATEGY_MINIMUM_MARGIN_RATE", "STRATEGY_TAYLOR_STEP"]
print(f'06_skew_strategy.ipynb 当前参数：')
for _parameter_name in _module_parameter_names:
    print(f'{_parameter_name} = {globals()[_parameter_name]!r}')


## 1. 依赖、参数与数据口径

输入来自 01、03、04 的 CSV；核心输出是每日 PnL、持仓、保证金、Taylor/Shapley 归因及诊断图。


In [ ]:
from __future__ import annotations
import itertools, math
from pathlib import Path
from typing import Dict, Tuple, TypeAlias
import numpy as np
import pandas as pd
from scipy.optimize import lsq_linear
from scipy.special import ndtr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.axes import Axes

DataRow: TypeAlias = pd.Series | tuple
ScalarOrArray: TypeAlias = float | np.ndarray

# 优先复用00--04在当前内核生成的对象；单独运行本Notebook时读取8.13自身CSV。
if 'option_forward_panel' in globals():
    OPT = option_forward_panel.copy()
else:
    _repo_path = Path(globals().get('REPO_FORWARD_OUTPUT_PATH', Path.cwd() / 'outputs' / '03_repo_forward'))
    OPT = pd.read_csv(_repo_path / 'option_forward_panel.csv')

# All tunable strategy parameters are defined in 00_config.ipynb.
# Explicit errors prevent a standalone Step-6 run from silently using stale defaults.
_REQUIRED_STRATEGY_PARAMETERS = [
    'STRATEGY_MAX_OPTIONS', 'STRATEGY_TARGET_SKEW_ABS',
    'STRATEGY_MONTHLY_SKEW_DIRECTION', 'STRATEGY_HEDGE_THETA',
    'STRATEGY_POSITION_BOUND',
    'STRATEGY_MAX_DAILY_TRADE_PER_LEG',
    'STRATEGY_MIN_OPTION_VOLUME', 'STRATEGY_USE_PREMIUM_MARGIN_FILTER',
    'STRATEGY_MIN_PREMIUM_MARGIN_RATIO', 'STRATEGY_MAX_ABS_LOG_MONEYNESS',
    'STRATEGY_MAX_MODEL_IV', 'STRATEGY_MIN_CANDIDATE_OPTIONS',
    'STRATEGY_MIN_DTE_DAYS', 'STRATEGY_MAX_CANDIDATE_EXPIRIES',
    'STRATEGY_RIDGE', 'STRATEGY_GROSS_POSITION_PENALTY',
    'STRATEGY_OPTION_MULTIPLIER', 'STRATEGY_OPTION_FEE_PER_CONTRACT',
    'STRATEGY_FUTURES_MULTIPLIER', 'STRATEGY_FUTURES_FEE_RATE',
    'STRATEGY_FUTURES_MARGIN_RATE',
    'STRATEGY_OPTION_MARGIN_RATE', 'STRATEGY_MINIMUM_MARGIN_RATE',
    'STRATEGY_TAYLOR_STEP',
]
_missing = [name for name in _REQUIRED_STRATEGY_PARAMETERS if name not in globals()]
if _missing:
    raise RuntimeError(
        'Run 00_config.ipynb before Step 6; missing strategy parameters: '
        + ', '.join(_missing)
    )
M = float(STRATEGY_OPTION_MULTIPLIER)
FEE = float(STRATEGY_OPTION_FEE_PER_CONTRACT)
FUTURES_MULTIPLIER = float(STRATEGY_FUTURES_MULTIPLIER)
FUTURES_FEE_RATE = float(STRATEGY_FUTURES_FEE_RATE)
FUTURES_MARGIN_RATE = float(STRATEGY_FUTURES_MARGIN_RATE)

# ========================== User interfaces ==========================
# Volatility model used for selection, pricing and attribution.
# Allowed values: 'QUADRATIC' and 'SVI'; defaults to the model selected in 00_config.
VOLATILITY_MODEL = str(globals().get('VOL_MODEL', 'QUADRATIC')).upper()
_model_tag = globals().get(
    'MODEL_SKEW_TAG',
    f'{VOLATILITY_MODEL}_SKEW-DERIVATIVE_CURVATURE-DERIVATIVE',
)
_project_path = Path(globals().get('PROJECT_PATH', Path.cwd())).resolve()
_output_root = Path(globals().get('OUTPUT_PATH', _project_path / 'outputs'))
VOLATILITY_PARAMETER_FILES = {
    VOLATILITY_MODEL: str(
        _output_root / f'04_volatility_model_{_model_tag}'
        / 'volatility_model_parameters.csv'
    ),
}

# Short aliases used by the implementation; values come from 00_config.ipynb.
MAX_OPTIONS = int(STRATEGY_MAX_OPTIONS)
MONTHLY_SKEW_DIRECTION = bool(STRATEGY_MONTHLY_SKEW_DIRECTION)
OPTION_MARGIN_RATE = float(STRATEGY_OPTION_MARGIN_RATE)
MINIMUM_MARGIN_RATE = float(STRATEGY_MINIMUM_MARGIN_RATE)
if float(STRATEGY_MIN_PREMIUM_MARGIN_RATIO) < 0:
    raise ValueError('STRATEGY_MIN_PREMIUM_MARGIN_RATIO must be non-negative')


## 2. 输入数据与波动率曲面

读取 01、03、04 的标准化面板，并把不同模型参数统一成局部 ATM、Skew、Curvature 因子。


In [ ]:
def load_volatility_parameters(model_name: str) -> pd.DataFrame:
    """读取并校验指定波动率模型参数，同时补齐利率字段。"""
    model_name = str(model_name).upper()
    if model_name not in VOLATILITY_PARAMETER_FILES:
        raise ValueError("VOLATILITY_MODEL must be 'QUADRATIC' or 'SVI'")
    path = Path(VOLATILITY_PARAMETER_FILES[model_name])
    if not path.exists():
        raise FileNotFoundError(
            f'{model_name} parameter CSV not found: {path}. '
            'Run 04_volatility_model.ipynb with the same VOL_MODEL first.'
        )
    parameters = pd.read_csv(path)
    required = ({'TRADE_DT','EXPIRY','TAU','a','b','c'} if model_name == 'QUADRATIC'
                else {'TRADE_DT','EXPIRY','TAU','a','b','rho','m','sigma'})
    missing = required-set(parameters.columns)
    if missing:
        raise ValueError(f'{model_name} parameter CSV missing columns: {sorted(missing)}')
    if 'MODEL' in parameters.columns:
        wrong = parameters.loc[~parameters['MODEL'].astype(str).str.upper().eq(model_name)]
        if not wrong.empty:
            raise ValueError(f'Parameter CSV contains rows not labelled {model_name}')
    parameters['TRADE_DT']=pd.to_datetime(parameters.TRADE_DT)
    parameters['EXPIRY']=pd.to_datetime(parameters.EXPIRY)
    if 'RISK_FREE_RATE' not in parameters.columns:
        rates = OPT.groupby(['TRADE_DT','EXPIRY'], as_index=False)[
            'RISK_FREE_RATE'
        ].first()
        parameters = parameters.merge(rates, on=['TRADE_DT','EXPIRY'], how='left')
        if parameters['RISK_FREE_RATE'].isna().any():
            raise ValueError('Unable to recover RISK_FREE_RATE from option panel')
    return parameters

OPT['TRADE_DT']=pd.to_datetime(OPT.TRADE_DT)
OPT['EXPIRY']=pd.to_datetime(OPT.EXPIRY)
PAR=load_volatility_parameters(VOLATILITY_MODEL)


In [ ]:
def surface_iv(model: DataRow, k: ScalarOrArray, tau: float | None = None) -> ScalarOrArray:
    """根据 Quadratic 或 SVI 参数计算给定对数价内度的隐含波动率。"""
    if VOLATILITY_MODEL == 'QUADRATIC':
        return np.maximum(model.a+model.b*np.asarray(k)+.5*model.c*np.asarray(k)**2, 1e-6)
    tau = float(model.TAU if tau is None else tau)
    x = np.asarray(k)-float(model.m)
    total_variance = float(model.a)+float(model.b)*(
        float(model.rho)*x+np.sqrt(x*x+float(model.sigma)**2)
    )
    return np.sqrt(np.maximum(total_variance, 1e-12)/max(tau, 1e-12))


In [ ]:
def surface_local_factors(model: DataRow, tau: float | None = None) -> np.ndarray:
    """返回曲面在 ATM 处的波动率、Skew 一阶导与 Curvature 二阶导。"""
    if VOLATILITY_MODEL == 'QUADRATIC':
        return np.array([float(model.a), float(model.b), float(model.c)])
    tau = float(model.TAU if tau is None else tau)
    a,b,rho,m,sigma = map(float,[model.a,model.b,model.rho,model.m,model.sigma])
    root=math.sqrt(m*m+sigma*sigma)
    w0=a+b*(-rho*m+root)
    w1=b*(rho-m/root)
    w2=b*sigma*sigma/(root**3)
    iv0=math.sqrt(max(w0,1e-12)/tau)
    skew=w1/(2*tau*iv0)
    curvature=w2/(2*tau*iv0)-w1*w1/(4*tau*tau*iv0**3)
    return np.array([iv0,skew,curvature])


## 3. Black–76 定价、风险与单腿归因

计算期权价值、五类组合风险，以及相邻交易日单腿的 Shapley 因子贡献。\n\nBlack–76：$V=e^{-r\tau}[F N(d_1)-K N(d_2)]$（Put 使用对应负号形式）。


In [ ]:
def price(F: float, K: float, v: float, t: float, r: float, cp: str) -> float:
    """使用 Black--76 公式计算期权价格。"""
    if t<=0 or v<=0: return max((F-K) if cp=='CALL' else (K-F),0)*math.exp(-r*max(t,0))
    z=(math.log(F/K)+.5*v*v*t)/(v*math.sqrt(t)); d=math.exp(-r*t)
    return d*((F*ndtr(z)-K*ndtr(z-v*math.sqrt(t))) if cp=='CALL' else (K*ndtr(-z+v*math.sqrt(t))-F*ndtr(-z)))


In [ ]:
#row 是 pandas.DataFrame 中的一行期权行情记录，通常由 DataFrame.itertuples() 生成的 namedtuple；
#它来自 04 输出的 option_iv_panel.csv（加载后形成 OPT），调用时一般是候选期权循环中的单行，
#包含 FORWARD、STRIKE、TAU、RISK_FREE_RATE、TYPE 等字段。
def risks(row: DataRow, p: DataRow) -> Tuple[np.ndarray, float, float]:
    """计算单只期权的 Forward Greeks、曲面风险向量、波动率和价内度。"""
    F,K,t,r,cp=row.FORWARD,row.STRIKE,row.TAU,row.RISK_FREE_RATE,row.TYPE
    k=math.log(K/F); v=float(surface_iv(p,k,t)); z=(math.log(F/K)+.5*v*v*t)/(v*math.sqrt(t)); disc=math.exp(-r*t); phi=math.exp(-z*z/2)/math.sqrt(2*math.pi)
    delta=disc*(ndtr(z) if cp=='CALL' else -ndtr(-z)); gamma=disc*phi/(F*v*math.sqrt(t)); vega=disc*F*phi*math.sqrt(t)
    skew=vega*k; curv=.5*vega*k*k
    # dV/dtau (not calendar theta); neutralizing this controls carry
    val=price(F,K,v,t,r,cp); dvdt=-r*val+disc*F*phi*v/(2*math.sqrt(t))
    return np.array([delta,gamma,vega,skew,curv,dvdt]),v,k


In [ ]:
def shapley_leg(q: float, row0: DataRow, row1: DataRow, p0: DataRow, p1: DataRow) -> Tuple[np.ndarray, float]:
    """用统一七因子状态对单腿全重估，并计算 Shapley PnL。"""
    names=['DELTA','SMILE_ROLL','ATM','SKEW','CURV','TAU','R']; n=len(names)
    a0,b0,c0=surface_local_factors(p0,row0.TAU)
    a1,b1,c1=surface_local_factors(p1,row1.TAU)
    F0,F1=float(row0.FORWARD),float(row1.FORWARD)
    k0,k1=math.log(float(row0.STRIKE)/F0),math.log(float(row0.STRIKE)/F1)
    old=np.array([F0,k0,a0,b0,c0,row0.TAU,row0.RISK_FREE_RATE],dtype=float)
    new=np.array([F1,k1,a1,b1,c1,row1.TAU,row1.RISK_FREE_RATE],dtype=float)
    def val(mask: int) -> float:
        """对七因子的给定新旧状态组合进行单腿重估。"""
        F_price,k_roll,atm,skew,curvature,tau,rate=[
            new[i] if mask&(1<<i) else old[i] for i in range(n)
        ]
        volatility=max(atm+skew*k_roll+.5*curvature*k_roll*k_roll,1e-6)
        return price(F_price,float(row0.STRIKE),volatility,tau,rate,row0.TYPE)
    cache={m:val(m) for m in range(1<<n)}; out=np.zeros(n)
    fact=math.factorial
    for i in range(n):
        for mask in range(1<<n):
            if mask&(1<<i): continue
            s=mask.bit_count(); w=fact(s)*fact(n-s-1)/fact(n)
            out[i]+=w*(cache[mask|(1<<i)]-cache[mask])
    return q*M*out, q*M*(cache[(1<<n)-1]-cache[0])


## 4. 选券、整数仓位与每日 PnL

逐日筛选 OTM 期权；默认进一步要求权利金金额与卖方单张保证金之比不低于 10%，再求解受限整数仓位，用 IM 期货对冲 Delta，并计算模型与市场 PnL。


In [ ]:
def run(target: float = -10000, bound: int = 100, minvol: int = 200, kmax: float = .22, ridge: float = 1e-5, max_options: int = MAX_OPTIONS,
        monthly_skew_direction: bool = MONTHLY_SKEW_DIRECTION,
        use_premium_margin_filter: bool = bool(STRATEGY_USE_PREMIUM_MARGIN_FILTER),
        min_premium_margin_ratio: float = float(STRATEGY_MIN_PREMIUM_MARGIN_RATIO),
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """逐日筛选期限与 OTM 期权、优化整数仓位、期货对冲并计算 PnL。"""
    dates=sorted(OPT.TRADE_DT.unique()); pars={(x.TRADE_DT,x.EXPIRY):x for x in PAR.itertuples() if not str(x.FIT_STATUS).startswith(('FAILED','INSUFFICIENT'))}
    pos=[]; pnl=[]
    previous_option_qty = {}
    previous_futures_qty = 0.0
    previous_futures_expiry = None
    for di,d in enumerate(dates[:-1]):
        dn=dates[di+1]; today=OPT[OPT.TRADE_DT.eq(d)]; tomorrow=OPT[OPT.TRADE_DT.eq(dn)]
        target_magnitude = abs(float(target))
        if monthly_skew_direction and pd.Timestamp(d).month in (6, 7):
            daily_target = target_magnitude if pd.Timestamp(d).month == 6 else -target_magnitude
        else:
            daily_target = float(target)
        exps=sorted(set(today.EXPIRY)&set(tomorrow.EXPIRY))
        exps=[e for e in exps if (d in [d]) and (d,e) in pars and (dn,e) in pars and (e-d).days >= int(STRATEGY_MIN_DTE_DAYS)]
        if not exps: continue
        # choose expiry with best-conditioned feasible hedge, usually front but search first 3
        best=None
        for e in exps[:int(STRATEGY_MAX_CANDIDATE_EXPIRIES)]:
            x=today[today.EXPIRY.eq(e)&today.VOLUME.ge(minvol)].merge(tomorrow[['CODE']],on='CODE')
            p0=pars[(d,e)]; rr=[]
            for row in x.itertuples():
                g,v,k=risks(row,p0)
                # OTM-only universe: put strikes at/below forward and call
                # strikes at/above forward.  ITM contracts are excluded.
                is_otm = ((row.TYPE == 'CALL' and row.STRIKE >= row.FORWARD)
                          or (row.TYPE == 'PUT' and row.STRIKE <= row.FORWARD))
                if not is_otm:
                    continue
                if use_premium_margin_filter:
                    call_otm = max(float(row.STRIKE) - float(row.SPOT), 0.0)
                    put_otm = max(float(row.SPOT) - float(row.STRIKE), 0.0)
                    otm_amount = M * (call_otm if row.TYPE == 'CALL' else put_otm)
                    premium_amount = M * float(row.PRICE)
                    margin_per_short = premium_amount + max(
                        OPTION_MARGIN_RATE * M * float(row.SPOT) - otm_amount,
                        MINIMUM_MARGIN_RATE * M * float(row.SPOT),
                    )
                    premium_margin_ratio = premium_amount / margin_per_short
                    if premium_margin_ratio < float(min_premium_margin_ratio):
                        continue
                if abs(k) <= kmax and v < float(STRATEGY_MAX_MODEL_IV): rr.append((row,g,k))
            if len(rr) < int(STRATEGY_MIN_CANDIDATE_OPTIONS): continue
            # Prefer broad OTM coverage; the leg-count interface controls the
            # maximum number sampled across the full k range.
            rr=sorted(rr,key=lambda z:z[2]); ids=np.unique(np.linspace(0,len(rr)-1,min(max_options,len(rr))).astype(int)); rr=[rr[i] for i in ids]
            full_risk_matrix=np.column_stack([z[1][[1,2,3,4,5]] for z in rr])
            # Theta 开关只控制 V_tau 是否进入优化；完整 exposure 始终保留并输出。
            optimization_rows=[0,1,2,3,4] if STRATEGY_HEDGE_THETA else [0,1,2,3]
            A=full_risk_matrix[optimization_rows,:]
            # normalized nuisance objective plus strong skew target; ridge suppresses gross
            target_vector=(np.array([0,0,daily_target,0,0]) if STRATEGY_HEDGE_THETA
                           else np.array([0,0,daily_target,0]))
            scales=np.maximum(np.linalg.norm(A,axis=1),1e-9); An=A/scales[:,None]; bn=target_vector/scales
            AA=np.vstack([An, math.sqrt(ridge)*np.eye(len(rr))]); bb=np.r_[bn,np.zeros(len(rr))]
            candidate_codes = [item[0].CODE for item in rr]
            trade_limit = float(STRATEGY_MAX_DAILY_TRADE_PER_LEG)
            previous = np.array([previous_option_qty.get(code, 0.0) for code in candidate_codes])
            lower = np.maximum(-float(bound), previous - trade_limit)
            upper = np.minimum(float(bound), previous + trade_limit)
            sol=lsq_linear(AA,bb,bounds=(lower,upper),lsmr_tol='auto').x
            # rescale toward target skew then clip
            if abs(A[2]@sol)>1e-9: sol=np.clip(sol*daily_target/(A[2]@sol), lower, upper)
            # Apply the configured option-lot convention.
            if STRATEGY_INTEGER_OPTION_POSITIONS:
                sol=np.clip(np.rint(sol), np.ceil(lower), np.floor(upper)).astype(int)
            nuisance=np.linalg.norm((A@sol-target_vector)/scales)
            score = nuisance + float(STRATEGY_GROSS_POSITION_PENALTY)*np.sum(abs(sol))
            if best is None or score<best[0]: best=(score,e,rr,sol,full_risk_matrix)
        if best is None: continue
        _,e,rr,q,full_risk_matrix=best; p0=pars[(d,e)]; p1=pars[(dn,e)]
        # Convert option Delta (yuan per index point) into the configured IM position.
        option_delta=M*sum(q[j]*rr[j][1][0] for j in range(len(rr)))
        raw_fut=-option_delta/FUTURES_MULTIPLIER
        fut=int(round(raw_fut)) if STRATEGY_INTEGER_FUTURES_POSITIONS else float(raw_fut)
        current_option_qty = {
            rr[j][0].CODE: (int(q[j]) if STRATEGY_INTEGER_OPTION_POSITIONS else float(q[j])) for j in range(len(rr)) if q[j] != 0
        }
        traded_codes = set(previous_option_qty) | set(current_option_qty)
        option_trade_qty = sum(
            abs(current_option_qty.get(code, 0) - previous_option_qty.get(code, 0))
            for code in traded_codes
        )
        # The final holding period also includes liquidation at its end date.
        final_close_qty = sum(abs(qty) for qty in current_option_qty.values()) \
            if di == len(dates)-2 else 0
        option_fee = FEE * (option_trade_qty + final_close_qty)
        previous_option_qty = current_option_qty
        fac=np.zeros(7); model=market=0
        for j,(row,g,k) in enumerate(rr):
            qj=q[j]
            if abs(qj)<1e-7: continue
            row1=tomorrow[tomorrow.CODE.eq(row.CODE)].iloc[0]
            f,m=shapley_leg(qj,row,pd.Series(row1),p0,p1); fac+=f; model+=m; market+=qj*M*(row1.PRICE-row.PRICE)
            pos.append([d,row.CODE,e,'OPTION',(int(qj) if STRATEGY_INTEGER_OPTION_POSITIONS else float(qj)),k,row.VOLUME,row.PRICE,
                        row.TYPE,row.STRIKE,row.SPOT])
        # Futures hedge belongs entirely to pure DELTA; Smile Roll applies only to options.
        df=float(tomorrow[tomorrow.EXPIRY.eq(e)].FORWARD.iloc[0]-today[today.EXPIRY.eq(e)].FORWARD.iloc[0])
        fut_pnl=fut*FUTURES_MULTIPLIER*df
        current_futures_price = float(today[today.EXPIRY.eq(e)].FORWARD.iloc[0])
        next_futures_price = float(tomorrow[tomorrow.EXPIRY.eq(e)].FORWARD.iloc[0])
        if previous_futures_expiry == e:
            # Same contract: charge only the net adjustment, e.g. 10 -> 2 means 8 lots.
            futures_trade_notional = (
                abs(fut - previous_futures_qty)
                * current_futures_price * FUTURES_MULTIPLIER
            )
        else:
            # Contract changed: close the previous expiry and open the new expiry.
            previous_close_notional = 0.0
            if previous_futures_expiry is not None and abs(previous_futures_qty) > 0:
                previous_rows = today[today.EXPIRY.eq(previous_futures_expiry)]
                if previous_rows.empty:
                    raise ValueError(
                        f'Missing current forward for previous futures expiry: '
                        f'{previous_futures_expiry}'
                    )
                previous_close_notional = (
                    abs(previous_futures_qty)
                    * float(previous_rows.FORWARD.iloc[0]) * FUTURES_MULTIPLIER
                )
            futures_trade_notional = (
                previous_close_notional
                + abs(fut) * current_futures_price * FUTURES_MULTIPLIER
            )
        futures_opening_fee = futures_trade_notional * FUTURES_FEE_RATE
        # The last holding period includes liquidation at t+1.
        futures_closing_fee = (
            abs(fut) * next_futures_price * FUTURES_MULTIPLIER * FUTURES_FEE_RATE
            if di == len(dates)-2 else 0.0
        )
        futures_fee = futures_opening_fee + futures_closing_fee
        fee=option_fee+futures_fee
        previous_futures_qty = fut
        previous_futures_expiry = e
        fac[0]+=fut_pnl; model+=fut_pnl; market+=fut_pnl
        pos.append([d,f'IM_{pd.Timestamp(e):%Y%m%d}',e,'FUTURE',fut,0.0,np.nan,float(today[today.EXPIRY.eq(e)].FORWARD.iloc[0]),np.nan,np.nan,float(today[today.EXPIRY.eq(e)].SPOT.iloc[0])])
        gross=sum(abs(q))
        # 组合 exposure 统一采用 q × 期权乘数 × 单份解析 Greek 的金额口径。
        cash_risk_exposures = M*(full_risk_matrix@q)
        pnl.append([dn,e,*fac,model,market,market-model,option_fee,futures_opening_fee,futures_closing_fee,futures_fee,fee,market-fee,gross,fut,*cash_risk_exposures])
    cols=['date','expiry','DELTA_PNL','SMILE_ROLL_PNL','ATM_PNL','SKEW_PNL','CURV_PNL','TAU_PNL','R_PNL','MODEL_PNL','MARKET_PNL','MARKET_NOISE_PNL','OPTION_FEE','FUTURES_OPENING_FEE','FUTURES_CLOSING_FEE','FUTURES_FEE','FEE','ACTUAL_PNL','GROSS_OPTION_POSITION','FUTURES_POSITION','GAMMA_EXPOSURE','ATMVOL_EXPOSURE','SKEW_EXPOSURE','CURVATURE_EXPOSURE','THETA_EXPOSURE']
    return pd.DataFrame(pnl,columns=cols),pd.DataFrame(pos,columns=['date','code','expiry','asset_type','qty','k','volume','price','option_type','strike','spot'])


## 5. 保证金

分别计算期权逐腿/逐日保证金与期货逐腿/逐日保证金。


In [ ]:
#输出是该期权腿六个状态变量（F、ATM、Skew、Curvature、期限、利率）的 Shapley PnL 贡献数组，以及该腿的新旧状态总模型 PnL。
def calculate_option_margin(positions: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """按交易所公式计算逐腿及逐日期权卖方保证金。"""
    option = positions.loc[positions['asset_type'].eq('OPTION')].copy()
    call_otm = np.maximum(option['strike'] - option['spot'], 0.0)
    put_otm = np.maximum(option['spot'] - option['strike'], 0.0)
    option['OTM_AMOUNT'] = np.where(
        option['option_type'].eq('CALL'), M * call_otm, M * put_otm
    )
    base = OPTION_MARGIN_RATE * M * option['spot'] - option['OTM_AMOUNT']
    minimum = MINIMUM_MARGIN_RATE * M * option['spot']
    option['MARGIN_PER_SHORT_CONTRACT'] = (
        M * option['price'] + np.maximum(base, minimum)
    )
    option['SHORT_CONTRACTS'] = np.maximum(-option['qty'], 0).astype(int)
    option['POSITION_MARGIN'] = (
        option['SHORT_CONTRACTS'] * option['MARGIN_PER_SHORT_CONTRACT']
    )
    daily = option.groupby('date', as_index=False).agg(
        OPTION_MARGIN=('POSITION_MARGIN', 'sum'),
        SHORT_OPTION_CONTRACTS=('SHORT_CONTRACTS', 'sum'),
        LONG_OPTION_CONTRACTS=('qty', lambda x: int(np.maximum(x, 0).sum())),
    )
    return option, daily


In [ ]:
def calculate_futures_margin(positions: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """按期货手数、乘数和保证金率计算逐腿及逐日保证金。"""
    futures = positions.loc[positions['asset_type'].eq('FUTURE')].copy()
    futures['FUTURES_CONTRACTS'] = futures['qty'].astype(int)
    futures['FUTURES_MULTIPLIER'] = FUTURES_MULTIPLIER
    futures['FUTURES_MARGIN_RATE'] = FUTURES_MARGIN_RATE
    futures['POSITION_MARGIN'] = (
        futures['FUTURES_CONTRACTS'].abs()
        * futures['price'] * FUTURES_MULTIPLIER * FUTURES_MARGIN_RATE
    )
    daily = futures.groupby('date', as_index=False).agg(
        FUTURES_MARGIN=('POSITION_MARGIN', 'sum'),
        FUTURES_CONTRACTS=('FUTURES_CONTRACTS', 'sum'),
        ABS_FUTURES_CONTRACTS=('FUTURES_CONTRACTS', lambda x: int(x.abs().sum())),
    )
    return futures, daily


## 6. 图形与持仓诊断

输出累计 PnL 图以及所选到期日的 Skew 期限段诊断。


In [ ]:
def plot_cumulative_pnl(pnl: pd.DataFrame, output_dir: Path | str) -> pd.DataFrame:
    """生成 Shapley 归因和实际 PnL 的累计图并返回累计数据。"""
    if plt is None:
        raise RuntimeError('matplotlib is required to generate PnL charts')
    output_dir = Path(output_dir)
    overview_dir = output_dir / 'overview'
    component_dir = output_dir / 'components'
    overview_dir.mkdir(parents=True, exist_ok=True)
    component_dir.mkdir(parents=True, exist_ok=True)

    data = pnl.copy()
    data['date'] = pd.to_datetime(data['date'])
    factor_columns = ['DELTA_PNL', 'SMILE_ROLL_PNL', 'ATM_PNL', 'SKEW_PNL', 'CURV_PNL', 'TAU_PNL', 'R_PNL']
    data['FEE_PNL'] = -data['FEE']
    data['MODEL_RESIDUAL'] = data['MODEL_PNL'] - data[factor_columns].sum(axis=1)
    data['TOTAL_RESIDUAL'] = (
        data['ACTUAL_PNL'] - data[factor_columns].sum(axis=1) - data['FEE_PNL']
    )
    data['MARKET_NOISE'] = data['TOTAL_RESIDUAL'] - data['MODEL_RESIDUAL']
    plot_columns = [
        'ACTUAL_PNL', 'MARKET_PNL', 'MODEL_PNL', *factor_columns,
        'MARKET_NOISE', 'MODEL_RESIDUAL', 'TOTAL_RESIDUAL', 'FEE_PNL',
    ]
    cumulative = data.set_index('date')[plot_columns].cumsum()

    def style_axis(ax: Axes, title: str, ylabel: str = 'Cumulative PnL') -> None:
        """统一设置累计 PnL 图的坐标轴样式。"""
        ax.axhline(0, color='#667085', linestyle='--', linewidth=.9)
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('Date')
        ax.set_ylabel(ylabel)
        ax.grid(color='#d9e0e8', linewidth=.7, alpha=.8)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))

    # Main account/model/market overview.
    fig, ax = plt.subplots(figsize=(13, 6.5))
    ax.plot(cumulative.index, cumulative['ACTUAL_PNL'], label='Actual PnL after fees', linewidth=2.5, color='#102a43')
    ax.plot(cumulative.index, cumulative['MARKET_PNL'], label='Market PnL before fees', linewidth=1.8, color='#2f80ed')
    ax.plot(cumulative.index, cumulative['MODEL_PNL'], label='Model PnL', linewidth=1.6, color='#27ae60')
    style_axis(ax, 'Cumulative Total PnL')
    ax.legend(frameon=True)
    fig.autofmt_xdate(); fig.tight_layout()
    fig.savefig(overview_dir/'01_cumulative_total_pnl.png', dpi=180, bbox_inches='tight')
    plt.close(fig)

    # All factor contributions.
    fig, ax = plt.subplots(figsize=(13, 6.5))
    colors = {'DELTA_PNL':'#2f80ed','SMILE_ROLL_PNL':'#00a6a6','ATM_PNL':'#9b51e0','SKEW_PNL':'#eb5757','CURV_PNL':'#f2994a','TAU_PNL':'#219653','R_PNL':'#7f8c8d'}
    for column in factor_columns:
        ax.plot(cumulative.index, cumulative[column], label=column, linewidth=2.4 if column == 'SKEW_PNL' else 1.4, color=colors[column])
    style_axis(ax, 'Cumulative PnL by Shapley Factor')
    ax.legend(ncol=3, frameon=True)
    fig.autofmt_xdate(); fig.tight_layout()
    fig.savefig(overview_dir/'02_cumulative_factor_pnl.png', dpi=180, bbox_inches='tight')
    plt.close(fig)

    # Residuals are kept separate from factor PnL.
    fig, ax = plt.subplots(figsize=(13, 6.5))
    ax.plot(cumulative.index, cumulative['MARKET_NOISE'], label='MARKET NOISE', linewidth=1.8, color='#c0392b')
    ax.plot(cumulative.index, cumulative['MODEL_RESIDUAL'], label='MODEL RESIDUAL', linewidth=1.8, color='#8e44ad')
    ax.plot(cumulative.index, cumulative['TOTAL_RESIDUAL'], label='TOTAL RESIDUAL', linewidth=2.2, color='#34495e')
    style_axis(ax, 'Cumulative Residual PnL')
    ax.legend(frameon=True)
    fig.autofmt_xdate(); fig.tight_layout()
    fig.savefig(overview_dir/'03_cumulative_residual_pnl.png', dpi=180, bbox_inches='tight')
    plt.close(fig)

    # Combined decomposition: factors + residual + fees = actual PnL.
    fig, ax = plt.subplots(figsize=(13, 6.5))
    decomposition = [*factor_columns, 'MODEL_RESIDUAL', 'MARKET_NOISE', 'FEE_PNL']
    for column in decomposition:
        ax.plot(cumulative.index, cumulative[column], label=column, linewidth=2.3 if column == 'SKEW_PNL' else 1.2)
    ax.plot(cumulative.index, cumulative['ACTUAL_PNL'], label='ACTUAL', linewidth=2.8, color='black')
    style_axis(ax, 'Cumulative PnL Decomposition')
    ax.legend(ncol=3, frameon=True)
    fig.autofmt_xdate(); fig.tight_layout()
    fig.savefig(overview_dir/'04_cumulative_pnl_decomposition.png', dpi=180, bbox_inches='tight')
    plt.close(fig)

    # Full all-in-one chart: totals, every factor, fees, and every residual.
    fig, ax = plt.subplots(figsize=(16, 8.5))
    factor_styles = {
        'DELTA_PNL': ('#2f80ed', 1.3), 'SMILE_ROLL_PNL': ('#00a6a6', 1.3),
        'ATM_PNL': ('#9b51e0', 1.3),
        'SKEW_PNL': ('#eb5757', 2.5), 'CURV_PNL': ('#f2994a', 1.3),
        'TAU_PNL': ('#219653', 1.3), 'R_PNL': ('#7f8c8d', 1.3),
    }
    for column, (color, width) in factor_styles.items():
        ax.plot(cumulative.index, cumulative[column], label=f'Factor: {column}',
                color=color, linewidth=width, alpha=.9)
    ax.plot(cumulative.index, cumulative['TOTAL_RESIDUAL'], label='TOTAL RESIDUAL',
            color='#34495e', linewidth=2.0, linestyle='--')
    ax.plot(cumulative.index, cumulative['FEE_PNL'], label='Fee PnL',
            color='#795548', linewidth=1.5, linestyle=':')
    ax.plot(cumulative.index, cumulative['MODEL_PNL'], label='Total: Model PnL',
            color='#27ae60', linewidth=2.4)
    ax.plot(cumulative.index, cumulative['MARKET_PNL'], label='Total: Market PnL',
            color='#1565c0', linewidth=2.7)
    ax.plot(cumulative.index, cumulative['ACTUAL_PNL'], label='Total: Actual PnL after fees',
            color='black', linewidth=3.2)
    style_axis(ax, 'All-in-One Cumulative PnL: Totals, Factors and Residuals')
    ax.legend(ncol=3, frameon=True, fontsize=9, loc='best')
    fig.autofmt_xdate(); fig.tight_layout()
    fig.savefig(overview_dir/'05_all_in_one_cumulative_pnl.png', dpi=200, bbox_inches='tight')
    plt.close(fig)

    # One chart per component, comparable to the 8.7/8.10 outputs.
    for column in plot_columns:
        fig, ax = plt.subplots(figsize=(10.5, 5))
        ax.plot(cumulative.index, cumulative[column], linewidth=2, color='#3568b8')
        style_axis(ax, f'Cumulative {column} PnL')
        fig.autofmt_xdate(); fig.tight_layout()
        fig.savefig(component_dir/f'{column.lower()}_cumulative_pnl.png', dpi=180, bbox_inches='tight')
        plt.close(fig)

    cumulative.reset_index().to_csv(output_dir/'cumulative_pnl.csv', index=False)
    return cumulative


In [ ]:
def plot_selected_expiry_skew_segments(positions: pd.DataFrame, model_parameters: pd.DataFrame, output_dir: Path | str) -> pd.DataFrame:
    """绘制每个持有期所选到期日的跨日 Skew 变化线段。"""
    if plt is None:
        raise RuntimeError('matplotlib is required to generate Skew segment chart')
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    pos = positions.loc[positions['asset_type'].eq('OPTION')].copy()
    pos['date'] = pd.to_datetime(pos['date']).dt.normalize()
    pos['expiry'] = pd.to_datetime(pos['expiry']).dt.normalize()
    selected = pos.groupby('date', as_index=False)['expiry'].first().sort_values('date')
    parameters = model_parameters.copy()
    parameters['TRADE_DT'] = pd.to_datetime(parameters['TRADE_DT']).dt.normalize()
    parameters['EXPIRY'] = pd.to_datetime(parameters['EXPIRY']).dt.normalize()
    lookup = {(row.TRADE_DT, row.EXPIRY): row for row in parameters.itertuples(index=False)}
    market_dates = sorted(parameters['TRADE_DT'].unique())
    next_date = {pd.Timestamp(market_dates[i]): pd.Timestamp(market_dates[i + 1])
                 for i in range(len(market_dates) - 1)}
    rows = []
    for row in selected.itertuples(index=False):
        start_date, expiry = pd.Timestamp(row.date), pd.Timestamp(row.expiry)
        end_date = next_date.get(start_date)
        if end_date is None or (start_date, expiry) not in lookup or (end_date, expiry) not in lookup:
            continue
        start_model, end_model = lookup[(start_date, expiry)], lookup[(end_date, expiry)]
        rows.append({
            'START_DATE': start_date, 'END_DATE': end_date, 'EXPIRY': expiry,
            'EXPIRY_CODE': f'{expiry.year % 100:02d}{expiry.month:02d}',
            'SKEW_START': float(surface_local_factors(start_model)[1]),
            'SKEW_END': float(surface_local_factors(end_model)[1]),
            'SKEW_CHANGE': float(surface_local_factors(end_model)[1]
                                 - surface_local_factors(start_model)[1]),
        })
    segments = pd.DataFrame(rows)
    segments.to_csv(output_dir/'selected_expiry_skew_segments.csv', index=False, date_format='%Y-%m-%d')

    codes = sorted(segments['EXPIRY_CODE'].unique())
    cmap = plt.get_cmap('tab10')
    colors = {code: cmap(i % 10) for i, code in enumerate(codes)}
    fig, ax = plt.subplots(figsize=(15, 7.5))
    labelled = set()
    for row in segments.itertuples(index=False):
        label = f'Expiry {row.EXPIRY_CODE}' if row.EXPIRY_CODE not in labelled else None
        ax.plot([row.START_DATE, row.END_DATE], [row.SKEW_START, row.SKEW_END],
                marker='o', markersize=4.2, linewidth=2.0,
                color=colors[row.EXPIRY_CODE], alpha=.9, label=label)
        labelled.add(row.EXPIRY_CODE)
    ax.axhline(0.0, color='#667085', linestyle='--', linewidth=.9)
    ax.set_title(f'Selected-Expiry ATM Derivative Skew ({VOLATILITY_MODEL}): '
                 'One Segment per Holding Period',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Trade Date')
    ax.set_ylabel(r'ATM IV Skew $\partial\sigma/\partial k\vert_{k=0}$')
    ax.grid(color='#d9e0e8', linewidth=.7, alpha=.8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    ax.legend(ncol=min(5, len(codes)), frameon=True, title='Selected expiry')
    fig.autofmt_xdate(); fig.tight_layout()
    fig.savefig(output_dir/'selected_expiry_skew_segments.png', dpi=200, bbox_inches='tight')
    plt.close(fig)
    return segments

# 全项目统一使用下一节定义的七因子状态。


## 7. 统一七因子 Shapley 与完整 Taylor

把原来的 Forward 状态拆为两个独立状态：

\[
Y=(F_{price},k_{roll},a,b,c,\tau,r),\qquad k_{roll}=\log(K/F).
\]

- `DELTA`：只改变 Black--76 定价中的 Forward，固定波动率曲面坐标；与 IM 期货对冲口径一致。
- `SMILE_ROLL`：固定定价 Forward，只改变期权在曲面上的横坐标 \(k\)。
- 其余状态分别表示 ATM、Skew、Curvature、期限和利率变化。

一阶 PnL 为 \(V_i\Delta Y_i\)，纯二阶为 \(\tfrac12V_{ii}(\Delta Y_i)^2\)，交叉项为 \(V_{ij}\Delta Y_i\Delta Y_j\)。交叉项保持独立，不分摊给一阶因子。

Exposure 表输出原始导数 \(V_i,V_{ii},V_{ij}\)，列名均带 `_EXPOSURE`；PnL 表输出导数乘以实际状态变化后的金额。


In [ ]:
TAYLOR_FACTOR_NAMES = ['DELTA','SMILE_ROLL','ATM','SKEW','CURV','TAU','R']


def taylor_state_vectors(
    row: DataRow, p0: DataRow, p1: DataRow,
) -> Tuple[np.ndarray, np.ndarray]:
    """构造七因子开仓状态与次日状态；F_price 与 k_roll 明确分离。"""
    a0, b0, c0 = surface_local_factors(p0, row.TAU)
    a1, b1, c1 = surface_local_factors(p1, p1.TAU)
    F0, F1 = float(row.FORWARD), float(p1.FORWARD)
    k0 = math.log(float(row.STRIKE)/F0)
    k1 = math.log(float(row.STRIKE)/F1)
    next_rate = float(getattr(p1, 'RISK_FREE_RATE', row.RISK_FREE_RATE))
    x0=np.array([F0,k0,a0,b0,c0,row.TAU,row.RISK_FREE_RATE],dtype=float)
    x1=np.array([F1,k1,a1,b1,c1,p1.TAU,next_rate],dtype=float)
    return x0, x1


def taylor_price_from_state(row: DataRow, state: np.ndarray) -> float:
    """按七因子状态定价；改变 F_price 时不会自动改变 k_roll。"""
    F_price,k_roll,atm,skew,curvature,tau,rate=map(float,state)
    volatility=max(atm+skew*k_roll+.5*curvature*k_roll*k_roll,1e-6)
    return price(F_price, float(row.STRIKE), volatility, tau, rate, row.TYPE)


def numerical_taylor_exposures(
    row: DataRow, p0: DataRow, p1: DataRow, relative_step: float = 1e-4,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """用实际状态单位的中心差分计算七个一阶和完整 7x7 Hessian exposure。"""
    x0, x1 = taylor_state_vectors(row, p0, p1)
    # 每个状态使用与自身尺度相适应的 bump；这是求导步长，不是实际市场变化。
    floors=np.array([1.,1e-3,1e-3,1e-3,1e-2,1/3650,1e-4])
    bumps = relative_step*np.maximum(np.abs(x0), floors)
    base = taylor_price_from_state(row, x0)
    gradient = np.zeros(len(x0)); hessian = np.zeros((len(x0), len(x0)))
    for i in range(len(x0)):
        ei = np.zeros(len(x0)); ei[i] = bumps[i]
        fp = taylor_price_from_state(row, x0 + ei)
        fm = taylor_price_from_state(row, x0 - ei)
        gradient[i] = (fp-fm)/(2*bumps[i])
        hessian[i, i] = (fp-2*base+fm)/(bumps[i]**2)
    for i in range(len(x0)):
        for j in range(i+1, len(x0)):
            ei = np.zeros(len(x0)); ej = np.zeros(len(x0))
            ei[i] = bumps[i]; ej[j] = bumps[j]
            cross = (
                taylor_price_from_state(row, x0+ei+ej)
                - taylor_price_from_state(row, x0+ei-ej)
                - taylor_price_from_state(row, x0-ei+ej)
                + taylor_price_from_state(row, x0-ei-ej)
            )/(4*bumps[i]*bumps[j])
            hessian[i, j] = hessian[j, i] = cross
    return gradient, hessian, x0, x1


def state_factor_shapley_leg(
    q: float, row: DataRow, p0: DataRow, p1: DataRow,
) -> Tuple[np.ndarray, float]:
    """复用全项目唯一的七因子 shapley_leg 实现。"""
    return shapley_leg(q, row, p1, p0, p1)


In [ ]:
def taylor_leg(
    q: float, row: DataRow, p0: DataRow, p1: DataRow, h: float = 1e-2,
) -> Tuple[np.ndarray, np.ndarray, float, Dict[str, float], Dict[str, float]]:
    """计算单腿七因子 Taylor PnL，并同时返回不含实际变化量的原始 exposure。"""
    gradient, hessian, x0, x1 = numerical_taylor_exposures(
        row, p0, p1, relative_step=h,
    )
    changes = x1-x0; scale = float(q)*M
    first_pnl = scale*gradient*changes
    allocated_second = first_pnl.copy()
    pnl_terms: Dict[str, float] = {}
    exposures: Dict[str, float] = {}
    for i, name in enumerate(TAYLOR_FACTOR_NAMES):
        pnl_terms[f'FIRST_{name}'] = first_pnl[i]
        pnl_terms[f'SECOND_{name}_{name}'] = .5*scale*hessian[i, i]*changes[i]**2
        exposures[f'FIRST_{name}_EXPOSURE'] = scale*gradient[i]
        exposures[f'SECOND_{name}_{name}_EXPOSURE'] = scale*hessian[i, i]
        allocated_second[i] += .5*scale*hessian[i, i]*changes[i]**2
    for i, left in enumerate(TAYLOR_FACTOR_NAMES):
        for j in range(i+1, len(TAYLOR_FACTOR_NAMES)):
            right = TAYLOR_FACTOR_NAMES[j]
            cross_pnl = scale*hessian[i, j]*changes[i]*changes[j]
            pnl_terms[f'CROSS_{left}_{right}'] = cross_pnl
            exposures[f'CROSS_{left}_{right}_EXPOSURE'] = scale*hessian[i, j]
            allocated_second[i] += .5*cross_pnl
            allocated_second[j] += .5*cross_pnl
    exact = scale*(taylor_price_from_state(row, x1)-taylor_price_from_state(row, x0))
    return first_pnl, allocated_second, exact, pnl_terms, exposures


def plot_individual_raw_exposures(
    exposure_data: pd.DataFrame, exposure_columns: list[str], output_dir: Path,
    title_prefix: str,
) -> None:
    """将每个 model Greek exposure 的原始值分别绘制为独立时间序列图。"""
    output_dir = Path(output_dir); output_dir.mkdir(parents=True, exist_ok=True)
    dates = pd.to_datetime(exposure_data['date'])
    for column in exposure_columns:
        figure, axis = plt.subplots(figsize=(13, 6.5))
        axis.plot(dates, exposure_data[column], linewidth=2, color='#1565c0')
        axis.axhline(0, color='#667085', linestyle='--', linewidth=.8)
        axis.set_title(f'{title_prefix}: {column}', fontweight='bold')
        axis.set_xlabel('Date'); axis.set_ylabel(f'Raw Model Greek Exposure: {column}')
        axis.grid(alpha=.3); axis.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
        figure.autofmt_xdate(); figure.tight_layout()
        filename = column.lower()+'.png'
        figure.savefig(output_dir/filename, dpi=200, bbox_inches='tight')
        plt.close(figure)


In [ ]:
def calculate_taylor_attribution(
    positions: pd.DataFrame, shapley_daily: pd.DataFrame,
    output_dir: Path | str, h: float = 1e-2,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """汇总统一七因子完整 Taylor、Shapley，以及全部一二阶 exposure。"""
    positions = positions.copy(); positions['date'] = pd.to_datetime(positions['date'])
    positions['expiry'] = pd.to_datetime(positions['expiry'])
    shapley_daily = shapley_daily.copy(); shapley_daily['date'] = pd.to_datetime(shapley_daily['date'])
    pars = {(x.TRADE_DT, x.EXPIRY): x for x in PAR.itertuples()
            if not str(x.FIT_STATUS).startswith(('FAILED', 'INSUFFICIENT'))}
    dates = sorted(OPT.TRADE_DT.unique())
    next_date = {pd.Timestamp(dates[i]): pd.Timestamp(dates[i+1]) for i in range(len(dates)-1)}
    records: list[Dict[str, float | pd.Timestamp]] = []
    exposure_records: list[Dict[str, float | pd.Timestamp]] = []
    for date, all_positions in positions.groupby('date'):
        date = pd.Timestamp(date); option_positions = all_positions[all_positions.asset_type.eq('OPTION')]
        if option_positions.empty or date not in next_date: continue
        end_date = next_date[date]; expiry = pd.Timestamp(option_positions.expiry.iloc[0])
        p0, p1 = pars[(date, expiry)], pars[(end_date, expiry)]
        old = OPT[(OPT.TRADE_DT.eq(date)) & OPT.EXPIRY.eq(expiry)].set_index('CODE')
        factor_count=len(TAYLOR_FACTOR_NAMES); linear=np.zeros(factor_count); second=np.zeros(factor_count); shapley_state=np.zeros(factor_count); explicit_terms: Dict[str, float] = {}
        portfolio_exposures: Dict[str, float] = {}
        for leg in option_positions.itertuples(index=False):
            first, full_second, _, leg_terms, leg_exposures = taylor_leg(
                leg.qty, old.loc[leg.code], p0, p1, h=h)
            linear += first; second += full_second
            leg_shapley, _ = state_factor_shapley_leg(leg.qty, old.loc[leg.code], p0, p1)
            shapley_state += leg_shapley
            for name, value in leg_terms.items(): explicit_terms[name] = explicit_terms.get(name, 0.0)+value
            for name, value in leg_exposures.items(): portfolio_exposures[name] = portfolio_exposures.get(name, 0.0)+value
        F0 = float(old.FORWARD.iloc[0]); F1 = float(p1.FORWARD)
        futures_contracts = float(all_positions.loc[all_positions.asset_type.eq('FUTURE'), 'qty'].iloc[0])
        futures_exposure = futures_contracts*FUTURES_MULTIPLIER
        futures_pnl = futures_exposure*(F1-F0)
        linear[0] += futures_pnl; second[0] += futures_pnl; shapley_state[0] += futures_pnl
        explicit_terms['FIRST_DELTA'] = explicit_terms.get('FIRST_DELTA', 0.0)+futures_pnl
        portfolio_exposures['FIRST_DELTA_EXPOSURE'] = portfolio_exposures.get('FIRST_DELTA_EXPOSURE', 0.0)+futures_exposure
        realized = shapley_daily.loc[shapley_daily.date.eq(end_date)].iloc[0]
        exact = float(realized.MODEL_PNL); fee_pnl = -float(realized.FEE)
        record: Dict[str, float | pd.Timestamp] = {
            'date': end_date, 'MODEL_PNL': exact, 'MARKET_PNL': float(realized.MARKET_PNL),
            'ACTUAL_PNL_AFTER_FEES': float(realized.ACTUAL_PNL), 'FEE_PNL': fee_pnl,
            **explicit_terms,
        }
        for i, name in enumerate(TAYLOR_FACTOR_NAMES):
            record[f'T1_{name}'] = linear[i]; record[f'T2_{name}'] = second[i]
            record[f'SH_{name}'] = shapley_state[i]
        for prefix, factor_sum in [('T1_', linear.sum()), ('T2_', second.sum()), ('SH_', shapley_state.sum())]:
            model_residual = exact-factor_sum
            total_residual = float(realized.ACTUAL_PNL)-factor_sum-fee_pnl
            record[prefix+'MODEL_RESIDUAL'] = model_residual
            record[prefix+'TOTAL_RESIDUAL'] = total_residual
            record[prefix+'MARKET_NOISE'] = total_residual-model_residual
        records.append(record)
        exposure_records.append({'date': date, 'expiry': expiry, **portfolio_exposures})
    comparison = pd.DataFrame(records); full_exposures = pd.DataFrame(exposure_records)
    output_dir = Path(output_dir); comparison.to_csv(output_dir/'t1_t2_shapley_pnl_comparison.csv', index=False)
    exposure_columns = [c for c in full_exposures.columns if c.endswith('_EXPOSURE')]
    full_exposures.to_csv(output_dir/'full_first_second_order_exposures.csv', index=False)
    figure_dir = output_dir/'figures'/'exposures'; figure_dir.mkdir(parents=True, exist_ok=True)
    plot_individual_raw_exposures(
        full_exposures, exposure_columns, figure_dir/'individual',
        'Full Taylor Raw Model Greek Exposure',
    )

    term_columns = [c for c in comparison.columns if c.startswith(('FIRST_', 'SECOND_', 'CROSS_'))]
    metadata = ['MODEL_PNL','MARKET_PNL','ACTUAL_PNL_AFTER_FEES','FEE_PNL',
                'T2_MODEL_RESIDUAL','T2_TOTAL_RESIDUAL','T2_MARKET_NOISE']
    explicit = comparison[['date', *term_columns, *metadata]].rename(columns={
        'T2_MODEL_RESIDUAL':'MODEL_RESIDUAL','T2_TOTAL_RESIDUAL':'TOTAL_RESIDUAL',
        'T2_MARKET_NOISE':'MARKET_NOISE'})
    values = [*term_columns,'MODEL_PNL','MARKET_PNL','ACTUAL_PNL_AFTER_FEES','FEE_PNL',
              'MODEL_RESIDUAL','TOTAL_RESIDUAL','MARKET_NOISE']
    explicit_output = pd.DataFrame({'date': explicit.date})
    for column in values:
        explicit_output[f'DAILY_{column}'] = explicit[column]
        explicit_output[f'CUMULATIVE_{column}'] = explicit[column].cumsum()
    explicit_output.to_csv(output_dir/'complete_taylor_pnl_daily_and_cumulative.csv', index=False)
    summary = pd.DataFrame([{
        'method': prefix.rstrip('_'),
        'signed_model_residual': comparison[prefix+'MODEL_RESIDUAL'].sum(),
        'absolute_model_residual': comparison[prefix+'MODEL_RESIDUAL'].abs().sum(),
        'skew_abs_share': comparison[prefix+'SKEW'].abs().sum()/comparison[[prefix+n for n in TAYLOR_FACTOR_NAMES]].abs().sum().sum(),
        'skew_pnl': comparison[prefix+'SKEW'].sum(),
    } for prefix in ('T1_','T2_','SH_')])
    summary.to_csv(output_dir/'attribution_method_summary.csv', index=False)

    # 恢复 8.12 已有的 T1/T2/Shapley 累积归因图；动态因子列表兼容 Smile Roll 开关。
    comparison['date'] = pd.to_datetime(comparison['date'])
    taylor_figure_dir = output_dir/'figures'/'taylor_comparison'
    taylor_figure_dir.mkdir(parents=True, exist_ok=True)

    def finish_taylor_plot(axis: Axes, title: str) -> None:
        """统一设置 Taylor 累积归因图的坐标、图例和日期格式。"""
        axis.axhline(0, color='#667085', linestyle='--', linewidth=.8)
        axis.set_title(title, fontweight='bold'); axis.set_xlabel('Date')
        axis.set_ylabel('Cumulative PnL'); axis.grid(alpha=.3)
        axis.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
        axis.legend(ncol=3, fontsize=8)
        axis.figure.autofmt_xdate(); axis.figure.tight_layout()

    for prefix, label in [('T1_', 'First-order Taylor'),
                          ('T2_', 'Second-order Taylor'), ('SH_', 'Shapley')]:
        figure, axis = plt.subplots(figsize=(13, 6.5))
        for name in TAYLOR_FACTOR_NAMES:
            axis.plot(comparison.date, comparison[prefix+name].cumsum(),
                      label=name, linewidth=2.3 if name == 'SKEW' else 1.3)
        for column, legend, style, color, width in [
            (prefix+'MODEL_RESIDUAL', 'MODEL RESIDUAL', '--', 'black', 2),
            (prefix+'MARKET_NOISE', 'MARKET NOISE', ':', '#c0392b', 1.6),
            (prefix+'TOTAL_RESIDUAL', 'TOTAL RESIDUAL', '-.', '#8e44ad', 1.8),
            ('MODEL_PNL', 'MODEL PNL', '-', '#34495e', 2.6),
            ('MARKET_PNL', 'MARKET PNL', '-', '#1565c0', 2.6),
            ('ACTUAL_PNL_AFTER_FEES', 'ACTUAL PNL AFTER FEES', '-', 'black', 3),
            ('FEE_PNL', 'FEE PNL', ':', '#795548', 1.5),
        ]:
            axis.plot(comparison.date, comparison[column].cumsum(), label=legend,
                      linestyle=style, color=color, linewidth=width)
        finish_taylor_plot(axis, f'Cumulative PnL Attribution - {label}')
        figure.savefig(taylor_figure_dir/f'{prefix.lower()}cumulative_attribution.png',
                       dpi=190, bbox_inches='tight'); plt.close(figure)

    figure, axis = plt.subplots(figsize=(13, 6.5))
    for prefix, label, color in [('T1_', 'Taylor 1 MODEL RESIDUAL', '#e74c3c'),
                                 ('T2_', 'Taylor 2 MODEL RESIDUAL', '#f39c12'),
                                 ('SH_', 'Shapley MODEL RESIDUAL', '#2c3e50')]:
        axis.plot(comparison.date, comparison[prefix+'MODEL_RESIDUAL'].cumsum(),
                  label=label, color=color, linewidth=2)
    finish_taylor_plot(axis, 'Cumulative MODEL RESIDUAL Comparison')
    figure.savefig(taylor_figure_dir/'residual_comparison.png', dpi=190,
                   bbox_inches='tight'); plt.close(figure)

    figure, axis = plt.subplots(figsize=(13, 6.5))
    for prefix, label, color in [('T1_', 'Taylor 1 Skew', '#9b59b6'),
                                 ('T2_', 'Taylor 2 Skew', '#e67e22'),
                                 ('SH_', 'Shapley Skew', '#c0392b')]:
        axis.plot(comparison.date, comparison[prefix+'SKEW'].cumsum(),
                  label=label, color=color, linewidth=2)
    finish_taylor_plot(axis, 'Cumulative Skew PnL Comparison')
    figure.savefig(taylor_figure_dir/'skew_comparison.png', dpi=190,
                   bbox_inches='tight'); plt.close(figure)
    return comparison, summary, full_exposures


## 8. Traditional Taylor：补充 Smile Roll 与专用 Exposure

Traditional Delta 保持与实际期货对冲一致：

\[
\Pi_{\Delta}=\left(M\sum_jq_j\Delta^{Black}_{F,j}+N_tM_F\right)\Delta F.
\]

新增的 Smile Roll 固定定价 Forward，只让 \(k_0\to k_1\)：

\[
\Pi_{roll}=M\sum_jq_j\nu_j(b+ck_j)\Delta k_j.
\]

Traditional exposure 表仅包含本归因实际使用的导数：Delta、Smile Roll、ATM、Skew、Curvature、Theta、Rho、Gamma、ATM Vanna 与 ATM Volga。


In [ ]:
TRADITIONAL_TAYLOR_COMPONENTS = [
    'DELTA_PNL','GAMMA_PNL','ATMVOL_PNL','SKEW_PNL','CURVATURE_PNL',
    'RHO_PNL','THETA_PNL','ATMVOL_VANNA_PNL','ATMVOL_VOLGA_PNL',
]
TRADITIONAL_TAYLOR_COMPONENTS.insert(1, 'SMILE_ROLL_PNL')


def traditional_greeks(row: DataRow, model: DataRow) -> Dict[str, float]:
    """计算 Traditional Taylor 使用的解析 exposure，包括独立 Smile Roll exposure。"""
    F,K,tau,r = map(float,[row.FORWARD,row.STRIKE,row.TAU,row.RISK_FREE_RATE])
    k=math.log(K/F); vol=float(surface_iv(model,k,tau)); sqrt_tau=math.sqrt(tau)
    d1=(math.log(F/K)+.5*vol*vol*tau)/(vol*sqrt_tau); d2=d1-vol*sqrt_tau
    discount=math.exp(-r*tau); phi=math.exp(-.5*d1*d1)/math.sqrt(2*math.pi)
    value=price(F,K,vol,tau,r,row.TYPE)
    delta=discount*(ndtr(d1) if row.TYPE=='CALL' else -ndtr(-d1))
    gamma=discount*phi/(F*vol*sqrt_tau); vega=discount*F*phi*sqrt_tau
    local_atm, local_skew, local_curvature = surface_local_factors(model,tau)
    return {
        'delta':delta,'smile_roll':vega*(local_skew+local_curvature*k),
        'gamma':gamma,'vega':vega,'skew':vega*k,'curvature':.5*vega*k*k,
        'rho':-tau*value,'theta_tau':-r*value+discount*F*phi*vol/(2*sqrt_tau),
        'atmvol_vanna':-discount*phi*d2/vol,'atmvol_volga':vega*d1*d2/vol,
    }


In [ ]:
def calculate_traditional_taylor_attribution(
    positions: pd.DataFrame, pnl_daily: pd.DataFrame, output_dir: Path | str,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """计算含 Smile Roll 的 Traditional PnL，并输出其专用 exposure 表与图。"""
    positions=positions.copy(); positions['date']=pd.to_datetime(positions.date)
    positions['expiry']=pd.to_datetime(positions.expiry)
    pnl_daily=pnl_daily.copy(); pnl_daily['date']=pd.to_datetime(pnl_daily.date)
    pars={(x.TRADE_DT,x.EXPIRY):x for x in PAR.itertuples()
          if not str(x.FIT_STATUS).startswith(('FAILED','INSUFFICIENT'))}
    dates=sorted(OPT.TRADE_DT.unique()); next_date={pd.Timestamp(dates[i]):pd.Timestamp(dates[i+1]) for i in range(len(dates)-1)}
    records=[]; exposure_records=[]
    exposure_map={
        'DELTA_EXPOSURE':'delta',
        'GAMMA_EXPOSURE':'gamma','ATMVOL_EXPOSURE':'vega','SKEW_EXPOSURE':'skew',
        'CURVATURE_EXPOSURE':'curvature','RHO_EXPOSURE':'rho','THETA_EXPOSURE':'theta_tau',
        'ATMVOL_VANNA_EXPOSURE':'atmvol_vanna','ATMVOL_VOLGA_EXPOSURE':'atmvol_volga'}
    exposure_map['SMILE_ROLL_EXPOSURE']='smile_roll'
    for date, all_positions in positions.groupby('date'):
        date=pd.Timestamp(date); option_positions=all_positions[all_positions.asset_type.eq('OPTION')]
        if option_positions.empty or date not in next_date: continue
        end_date=next_date[date]; expiry=pd.Timestamp(option_positions.expiry.iloc[0])
        p0,p1=pars[(date,expiry)],pars[(end_date,expiry)]
        old=OPT[(OPT.TRADE_DT.eq(date))&OPT.EXPIRY.eq(expiry)].set_index('CODE')
        new=OPT[(OPT.TRADE_DT.eq(end_date))&OPT.EXPIRY.eq(expiry)].set_index('CODE')
        components={name:0.0 for name in TRADITIONAL_TAYLOR_COMPONENTS}
        exposures={name:0.0 for name in exposure_map}
        for leg in option_positions.itertuples(index=False):
            row0,row1=old.loc[leg.code],new.loc[leg.code]; g=traditional_greeks(row0,p0)
            dF=float(row1.FORWARD-row0.FORWARD); k0=math.log(float(row0.STRIKE)/float(row0.FORWARD))
            k1=math.log(float(row0.STRIKE)/float(row1.FORWARD)); dk=k1-k0
            f0=surface_local_factors(p0,row0.TAU); f1=surface_local_factors(p1,row1.TAU)
            da,db,dc=(f1-f0).astype(float); dr=float(row1.RISK_FREE_RATE-row0.RISK_FREE_RATE)
            dtau=float(row1.TAU-row0.TAU); scale=float(leg.qty)*M
            components['DELTA_PNL']+=scale*g['delta']*dF
            components['SMILE_ROLL_PNL']+=scale*g['smile_roll']*dk
            components['GAMMA_PNL']+=.5*scale*g['gamma']*dF*dF
            components['ATMVOL_PNL']+=scale*g['vega']*da; components['SKEW_PNL']+=scale*g['skew']*db
            components['CURVATURE_PNL']+=scale*g['curvature']*dc; components['RHO_PNL']+=scale*g['rho']*dr
            components['THETA_PNL']+=scale*g['theta_tau']*dtau
            components['ATMVOL_VANNA_PNL']+=scale*g['atmvol_vanna']*dF*da
            components['ATMVOL_VOLGA_PNL']+=.5*scale*g['atmvol_volga']*da*da
            for output_name, greek_name in exposure_map.items(): exposures[output_name]+=scale*g[greek_name]
        futures_contracts=float(all_positions.loc[all_positions.asset_type.eq('FUTURE'),'qty'].iloc[0])
        dF=float(new.FORWARD.iloc[0]-old.FORWARD.iloc[0]); futures_exposure=futures_contracts*FUTURES_MULTIPLIER
        components['DELTA_PNL']+=futures_exposure*dF; exposures['DELTA_EXPOSURE']+=futures_exposure
        realized=pnl_daily.loc[pnl_daily.date.eq(end_date)].iloc[0]; exact=float(realized.MODEL_PNL)
        greek_sum=sum(components.values()); fee_pnl=-float(realized.FEE)
        model_residual=exact-greek_sum; total_residual=float(realized.ACTUAL_PNL)-greek_sum-fee_pnl
        records.append({'date':end_date,**components,'MODEL_RESIDUAL':model_residual,
                        'MARKET_NOISE':total_residual-model_residual,'TOTAL_RESIDUAL':total_residual,
                        'FEE_PNL':fee_pnl,'MODEL_PNL':exact,'MARKET_PNL':float(realized.MARKET_PNL),
                        'ACTUAL_PNL_AFTER_FEES':float(realized.ACTUAL_PNL)})
        exposure_records.append({'date':date,'expiry':expiry,**exposures})
    attribution=pd.DataFrame(records); traditional_exposures=pd.DataFrame(exposure_records); output_dir=Path(output_dir)
    attribution.to_csv(output_dir/'traditional_taylor_attribution.csv',index=False)
    traditional_exposures.to_csv(output_dir/'traditional_taylor_exposures.csv',index=False)
    exposure_columns=[c for c in traditional_exposures.columns if c.endswith('_EXPOSURE')]
    figure_dir=output_dir/'figures'/'traditional_taylor'; figure_dir.mkdir(parents=True,exist_ok=True)
    plot_individual_raw_exposures(
        traditional_exposures, exposure_columns, figure_dir/'exposures',
        'Traditional Taylor Raw Model Greek Exposure',
    )
    denominator=attribution[TRADITIONAL_TAYLOR_COMPONENTS].abs().sum().sum()
    summary=pd.DataFrame([{'method':'TRADITIONAL_TAYLOR',
        'signed_model_residual':attribution.MODEL_RESIDUAL.sum(),
        'absolute_model_residual':attribution.MODEL_RESIDUAL.abs().sum(),
        'skew_abs_share':attribution.SKEW_PNL.abs().sum()/denominator if denominator else np.nan,
        'skew_pnl':attribution.SKEW_PNL.sum()}])
    summary.to_csv(output_dir/'traditional_taylor_summary.csv',index=False)
    attribution['date']=pd.to_datetime(attribution.date); fig,ax=plt.subplots(figsize=(16,8))
    for component in TRADITIONAL_TAYLOR_COMPONENTS:
        ax.plot(attribution.date,attribution[component].cumsum(),label=component,
                linewidth=2.3 if component=='SKEW_PNL' else 1.2)
    for column,label,style,color,width in [
        ('MODEL_RESIDUAL','MODEL RESIDUAL','--','black',2),('MARKET_NOISE','MARKET NOISE',':','#c0392b',1.7),
        ('TOTAL_RESIDUAL','TOTAL RESIDUAL','-.','#8e44ad',1.8),('FEE_PNL','FEE PNL',':','#795548',1.5),
        ('MODEL_PNL','MODEL PNL','-','#34495e',2.6),('MARKET_PNL','MARKET PNL','-','#1565c0',2.6),
        ('ACTUAL_PNL_AFTER_FEES','ACTUAL PNL AFTER FEES','-','black',3)]:
        ax.plot(attribution.date,attribution[column].cumsum(),label=label,linestyle=style,color=color,linewidth=width)
    ax.axhline(0,color='#667085',linestyle='--',linewidth=.8); ax.set_title('Cumulative Traditional Taylor PnL Attribution')
    ax.set_xlabel('Date');ax.set_ylabel('Cumulative PnL');ax.grid(alpha=.3);ax.legend(ncol=3,fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'));fig.autofmt_xdate();fig.tight_layout()
    fig.savefig(figure_dir/'traditional_taylor_cumulative_attribution.png',dpi=200,bbox_inches='tight');plt.close(fig)
    return attribution,summary,traditional_exposures


## 9. 执行与落盘

按既有顺序运行策略、保证金、图形及三套归因，并保存全部结果。


In [ ]:
if __name__=='__main__':
    output_dir = Path(globals().get(
        'SKEW_STRATEGY_OUTPUT_PATH',
        _output_root / f'06_skew_strategy_{_model_tag}',
    ))
    output_dir.mkdir(parents=True, exist_ok=True)
    # Frozen successful in-sample parameters; no parameter search is performed here.
    p, z = run(
        target=float(STRATEGY_TARGET_SKEW_ABS),
        bound=int(STRATEGY_POSITION_BOUND),
        minvol=int(STRATEGY_MIN_OPTION_VOLUME),
        kmax=float(STRATEGY_MAX_ABS_LOG_MONEYNESS),
        ridge=float(STRATEGY_RIDGE),
        max_options=MAX_OPTIONS,
        monthly_skew_direction=MONTHLY_SKEW_DIRECTION,
        use_premium_margin_filter=bool(STRATEGY_USE_PREMIUM_MARGIN_FILTER),
        min_premium_margin_ratio=float(STRATEGY_MIN_PREMIUM_MARGIN_RATIO),
    )
    p.to_csv(output_dir/'shapley_attribution.csv', index=False)
    z.to_csv(output_dir/'positions.csv', index=False)
    option_margin_by_leg, option_margin_daily = calculate_option_margin(z)
    futures_margin_by_leg, futures_margin_daily = calculate_futures_margin(z)
    option_margin_by_leg.to_csv(output_dir/'option_margin_by_leg.csv', index=False)
    option_margin_daily.to_csv(output_dir/'option_margin_daily.csv', index=False)
    futures_margin_by_leg.to_csv(output_dir/'futures_margin_by_leg.csv', index=False)
    futures_margin_daily.to_csv(output_dir/'futures_margin_daily.csv', index=False)
    margin_daily = option_margin_daily.merge(futures_margin_daily, on='date', how='outer')
    margin_daily[['OPTION_MARGIN', 'FUTURES_MARGIN']] = (
        margin_daily[['OPTION_MARGIN', 'FUTURES_MARGIN']].fillna(0.0)
    )
    margin_daily['TOTAL_MARGIN'] = (
        margin_daily['OPTION_MARGIN'] + margin_daily['FUTURES_MARGIN']
    )
    margin_daily.to_csv(output_dir/'total_margin_daily.csv', index=False)
    cumulative = plot_cumulative_pnl(p, output_dir/'figures')
    skew_segments = plot_selected_expiry_skew_segments(
        z, PAR, output_dir/'figures'/'skew_diagnostics'
    )
    taylor_comparison, attribution_summary, full_taylor_exposures = calculate_taylor_attribution(
        z, p, output_dir, h=float(STRATEGY_TAYLOR_STEP)
    )
    traditional_taylor, traditional_taylor_summary, traditional_taylor_exposures = (
        calculate_traditional_taylor_attribution(z, p, output_dir)
    )
    factor_columns = ['DELTA_PNL', 'SMILE_ROLL_PNL', 'ATM_PNL', 'SKEW_PNL', 'CURV_PNL', 'TAU_PNL', 'R_PNL']
    share = p.SKEW_PNL.abs().sum()/p[factor_columns].abs().sum().sum()
    first_date = pd.Timestamp(z['date'].min())
    first_options = z.loc[
        z['asset_type'].eq('OPTION') & pd.to_datetime(z['date']).eq(first_date)
    ]
    first_net_opening_cost = float(
        (M * first_options['qty'] * first_options['price']).sum()
    )
    first_premium_cash_flow = -first_net_opening_cost
    first_option_opening_fee = float(FEE * first_options['qty'].abs().sum())
    first_futures_opening_fee = float(p['FUTURES_OPENING_FEE'].iloc[0])
    first_opening_fee = first_option_opening_fee + first_futures_opening_fee
    first_option_margin = float(
        margin_daily.loc[pd.to_datetime(margin_daily['date']).eq(first_date),
                         'OPTION_MARGIN'].iloc[0]
    )
    first_futures_margin = float(
        margin_daily.loc[pd.to_datetime(margin_daily['date']).eq(first_date),
                         'FUTURES_MARGIN'].iloc[0]
    )
    first_total_margin = first_option_margin + first_futures_margin
    first_capital_payment = first_total_margin + first_net_opening_cost + first_opening_fee
    final_pnl_after_fees = float(p.ACTUAL_PNL.sum())
    r_first = (
        final_pnl_after_fees / first_capital_payment
        if not np.isclose(first_capital_payment, 0.0) else np.nan
    )
    # Capital required on each opening date. PnL dated on or before that date
    # has already been realized and can support the newly opened position.
    option_opening = z.loc[z['asset_type'].eq('OPTION')].copy()
    option_opening['date'] = pd.to_datetime(option_opening['date'])
    option_qty_panel = option_opening.pivot_table(
        index='date', columns='code', values='qty', aggfunc='sum', fill_value=0
    ).sort_index()
    option_net_trades = option_qty_panel.diff().fillna(option_qty_panel)
    net_opening_fee = (
        FEE * option_net_trades.abs().sum(axis=1)
    ).rename('OPTION_OPENING_FEE').reset_index()
    pnl_dates = sorted(pd.to_datetime(p['date']).unique())
    opening_dates = sorted(pd.to_datetime(z['date']).unique())
    if len(pnl_dates) != len(opening_dates):
        raise ValueError('PnL dates and opening dates are not one-to-one')
    futures_opening_fee = pd.DataFrame({
        'date': opening_dates,
        'FUTURES_OPENING_FEE': p.sort_values('date')['FUTURES_OPENING_FEE'].to_numpy(),
    })
    net_opening_fee = net_opening_fee.merge(
        futures_opening_fee, on='date', how='outer'
    ).fillna(0.0)
    net_opening_fee['OPENING_FEE'] = (
        net_opening_fee['OPTION_OPENING_FEE']
        + net_opening_fee['FUTURES_OPENING_FEE']
    )
    opening_cash = option_opening.assign(
        PREMIUM_COST=M * option_opening['qty'] * option_opening['price'],
    ).groupby('date', as_index=False).agg(
        NET_OPENING_PREMIUM_COST=('PREMIUM_COST', 'sum'),
    )
    opening_cash = opening_cash.merge(net_opening_fee, on='date', how='left')
    capital_daily = margin_daily.copy()
    capital_daily['date'] = pd.to_datetime(capital_daily['date'])
    capital_daily = capital_daily.merge(opening_cash, on='date', how='left')
    realized = p[['date', 'ACTUAL_PNL']].copy()
    realized['date'] = pd.to_datetime(realized['date'])
    realized = realized.groupby('date')['ACTUAL_PNL'].sum().sort_index().cumsum()
    capital_daily['CUMULATIVE_PNL_BEFORE_OPENING'] = (
        realized.reindex(capital_daily['date'], method='ffill').fillna(0.0).to_numpy()
    )
    capital_daily['REQUIRED_INITIAL_CAPITAL'] = (
        capital_daily['TOTAL_MARGIN']
        + capital_daily['NET_OPENING_PREMIUM_COST']
        + capital_daily['OPENING_FEE']
        - capital_daily['CUMULATIVE_PNL_BEFORE_OPENING']
    )
    recommended_capital = float(capital_daily['REQUIRED_INITIAL_CAPITAL'].max())
    recommended_capital_date = pd.Timestamp(
        capital_daily.loc[capital_daily['REQUIRED_INITIAL_CAPITAL'].idxmax(), 'date']
    )
    r_recommended = (
        final_pnl_after_fees / recommended_capital
        if not np.isclose(recommended_capital, 0.0) else np.nan
    )
    capital_daily.to_csv(output_dir/'capital_requirement_daily.csv', index=False)
    max_margin = float(margin_daily['TOTAL_MARGIN'].max())
    max_margin_date = pd.Timestamp(
        margin_daily.loc[margin_daily['TOTAL_MARGIN'].idxmax(), 'date']
    )
    print(f'Days: {len(p)}')
    print(f'Skew absolute PnL share: {share:.2%}')
    print(f'Cumulative Skew PnL: {p.SKEW_PNL.sum():,.2f}')
    print(f'Cumulative Actual PnL: {p.ACTUAL_PNL.sum():,.2f}')
    print('Taylor/Shapley attribution summary:')
    print(attribution_summary.to_string(index=False))
    print('Traditional Greeks Taylor attribution summary:')
    print(traditional_taylor_summary.to_string(index=False))
    print(f'Skew holding-period segments: {len(skew_segments)}')
    print(f'Maximum option-leg interface: {MAX_OPTIONS}')
    print(f'Volatility model interface: {VOLATILITY_MODEL}')
    print(f'Volatility parameter CSV: {VOLATILITY_PARAMETER_FILES[VOLATILITY_MODEL]}')
    print(f'Monthly skew direction interface: {MONTHLY_SKEW_DIRECTION}')
    if MONTHLY_SKEW_DIRECTION:
        print('Skew target schedule: June +10,000; July -10,000')
    print('Option universe: OTM only')
    print(f'Premium/margin candidate filter: {bool(STRATEGY_USE_PREMIUM_MARGIN_FILTER)}')
    if STRATEGY_USE_PREMIUM_MARGIN_FILTER:
        print(f'Minimum premium/margin ratio: {float(STRATEGY_MIN_PREMIUM_MARGIN_RATIO):.2%}')
    print(f'Futures contract multiplier: {FUTURES_MULTIPLIER:,.0f}')
    print(f'Futures margin rate: {FUTURES_MARGIN_RATE:.2%}')
    print(f'First-day option margin: {first_option_margin:,.2f}')
    print(f'First-day futures margin: {first_futures_margin:,.2f}')
    print(f'First-day total margin: {first_total_margin:,.2f}')
    print(f'First-day net opening premium cost: {first_net_opening_cost:,.2f} '
          '(negative means net premium received)')
    print(f'First-day premium cash flow: {first_premium_cash_flow:,.2f}')
    print(f'First-day option opening fee: {first_option_opening_fee:,.2f}')
    print(f'First-day futures opening fee: {first_futures_opening_fee:,.2f}')
    print(f'First-day total opening fee: {first_opening_fee:,.2f}')
    print('First-day total margin + opening premium cost + fee: '
          f'{first_capital_payment:,.2f}')
    print(f'Maximum total margin: {max_margin:,.2f} on {max_margin_date:%Y-%m-%d}')
    print(f'Figures: {output_dir / "figures"}')
    print(f'Final cumulative PnL after transaction fees: {final_pnl_after_fees:,.2f}')
    print(f'R_first: {r_first:.2%}')
    print('Recommended required initial capital: '
          f'{recommended_capital:,.2f} on {recommended_capital_date:%Y-%m-%d}')
    print(f'R_recommended: {r_recommended:.2%}')
